# Minimal Gap Forecast Evaluation

Compare trimmed mean, fixed gap add-backs, core PCE, and an expanding-window estimated gap add-back as forecasts of average headline PCE inflation over the next 6, 9, 12, and 24 months.

The notebook reports the main forecast comparison for positive-skew months and all months, then adds three direct positive-skew tests:
1. What caused the contemporaneous headline-minus-trimmed gap?
2. Which gap components predict future headline inflation relative to trimmed mean?
3. Do estimated demand, supply, and ambiguous adjustment weights improve pseudo-out-of-sample forecasts?


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "Data_Final.xlsx").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

DATA_PATH = ROOT / "Data_Final.xlsx"
CORE_PATH = ROOT / "outputs/gap_signal_analysis_20260621/core_pce_index.csv"

HORIZONS = [6, 9, 12, 24]
COMPONENT_COLS = ["Demand_Gap", "Supply_Gap", "Ambiguous_Gap"]
COMPONENT_NAMES = ["Demand", "Supply", "Ambiguous"]
MIN_TRAIN = 24


def format_table(frame):
    return frame.to_string(index=False, float_format=lambda x: f"{x:0.3f}")


def evaluate(sample, measures):
    rows = []
    for h in HORIZONS:
        target = f"Headline_next_{h}m_avg"
        for label, col in measures.items():
            use = sample & df[col].notna() & df[target].notna()
            error = df.loc[use, col] - df.loc[use, target]
            rows.append(
                {
                    "Horizon": h,
                    "Measure": label,
                    "N": int(error.size),
                    "Bias": error.mean(),
                    "MAE": error.abs().mean(),
                    "RMSE": np.sqrt(np.mean(error**2)),
                }
            )
    return pd.DataFrame(rows).sort_values(["Horizon", "RMSE"]).reset_index(drop=True)


def ols_hac_fit(y, x, term_names, lag):
    y = np.asarray(y, dtype=float)
    x = np.asarray(x, dtype=float)
    if x.ndim == 1:
        x = x.reshape(-1, 1)

    valid = np.isfinite(y) & np.all(np.isfinite(x), axis=1)
    y = y[valid]
    x = x[valid]
    n = len(y)

    design = np.column_stack([np.ones(n), x])
    terms = ["Intercept"] + list(term_names)
    k = design.shape[1]
    if n <= k:
        return {"n": n, "terms": terms, "coef": np.full(k, np.nan), "t": np.full(k, np.nan), "r2": np.nan}

    beta = np.linalg.lstsq(design, y, rcond=None)[0]
    resid = y - design @ beta
    xtx_inv = np.linalg.pinv(design.T @ design)

    s = np.zeros((k, k))
    for t in range(n):
        s += resid[t] ** 2 * np.outer(design[t], design[t])

    max_lag = min(int(lag), n - 1)
    for ell in range(1, max_lag + 1):
        weight = 1 - ell / (max_lag + 1)
        gamma = np.zeros((k, k))
        for t in range(ell, n):
            gamma += resid[t] * resid[t - ell] * np.outer(design[t], design[t - ell])
        s += weight * (gamma + gamma.T)

    cov = xtx_inv @ s @ xtx_inv
    se = np.sqrt(np.maximum(np.diag(cov), 0))
    t_stat = np.divide(beta, se, out=np.full_like(beta, np.nan), where=se > 0)
    sst = np.sum((y - y.mean()) ** 2)
    r2 = 1 - np.sum(resid**2) / sst if sst > 0 else np.nan
    return {"n": n, "terms": terms, "coef": beta, "t": t_stat, "r2": r2}


def fit_no_intercept(y, x):
    y = np.asarray(y, dtype=float)
    x = np.asarray(x, dtype=float)
    valid = np.isfinite(y) & np.all(np.isfinite(x), axis=1)
    return np.linalg.lstsq(x[valid], y[valid], rcond=None)[0]


In [2]:
df = pd.read_excel(DATA_PATH)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

core = pd.read_csv(CORE_PATH)
date_col = "observation_date" if "observation_date" in core.columns else core.columns[0]
value_col = "PCEPILFE" if "PCEPILFE" in core.columns else core.columns[-1]
core[date_col] = pd.to_datetime(core[date_col])
core[value_col] = pd.to_numeric(core[value_col], errors="coerce")
core["Core_PCE"] = 100 * (core[value_col] / core[value_col].shift(12) - 1)
core = core[[date_col, "Core_PCE"]].rename(columns={date_col: "Date"})

df = df.merge(core, on="Date", how="left")
df["TM_plus_Demand"] = df["Trimmed_Mean_PCE"] + df["Demand_Gap"]
df["TM_plus_Supply"] = df["Trimmed_Mean_PCE"] + df["Supply_Gap"]
df["TM_plus_Demand_Supply"] = df["Trimmed_Mean_PCE"] + df["Demand_Gap"] + df["Supply_Gap"]

headline = df["Headline_PCE"].to_numpy(dtype=float)
for h in HORIZONS:
    df[f"Headline_next_{h}m_avg"] = [
        np.nanmean(headline[i + 1 : i + h + 1]) if i + h < len(df) else np.nan
        for i in range(len(df))
    ]

positive_skew_sample = df["Kelly_Skewness_12m_avg"] > 0
all_months_sample = pd.Series(True, index=df.index)

measures = {
    "Trimmed mean": "Trimmed_Mean_PCE",
    "Trimmed mean + demand gap": "TM_plus_Demand",
    "Trimmed mean + supply gap": "TM_plus_Supply",
    "Trimmed mean + demand + supply gaps": "TM_plus_Demand_Supply",
    "Core PCE": "Core_PCE",
}

print(f"Data range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"Positive-skew months before horizon availability filters: {int(positive_skew_sample.sum())}")
print(f"All months before horizon availability filters: {int(all_months_sample.sum())}")


Data range: 1980-12-01 to 2026-04-01
Positive-skew months before horizon availability filters: 64
All months before horizon availability filters: 545


## Positive-skew months


In [3]:
positive_results = evaluate(positive_skew_sample, measures)
print(format_table(positive_results))


 Horizon                             Measure  N   Bias   MAE  RMSE
       6                            Core PCE 59 -0.750 0.981 1.168
       6 Trimmed mean + demand + supply gaps 59 -0.008 0.963 1.222
       6           Trimmed mean + demand gap 59 -0.629 1.024 1.329
       6           Trimmed mean + supply gap 59 -0.589 1.118 1.450
       6                        Trimmed mean 59 -1.210 1.443 1.798
       9                            Core PCE 59 -0.595 1.023 1.250
       9           Trimmed mean + demand gap 59 -0.474 1.121 1.495
       9 Trimmed mean + demand + supply gaps 59  0.147 1.200 1.511
       9           Trimmed mean + supply gap 59 -0.434 1.261 1.650
       9                        Trimmed mean 59 -1.055 1.452 1.879
      12                            Core PCE 59 -0.412 1.065 1.353
      12           Trimmed mean + demand gap 59 -0.292 1.254 1.654
      12 Trimmed mean + demand + supply gaps 59  0.330 1.433 1.773
      12           Trimmed mean + supply gap 59 -0.251 1.431 1

In [4]:
positive_rmse_table = positive_results.pivot(index="Horizon", columns="Measure", values="RMSE")
print(positive_rmse_table.to_string(float_format=lambda x: f"{x:0.3f}"))


Measure  Core PCE  Trimmed mean  Trimmed mean + demand + supply gaps  Trimmed mean + demand gap  Trimmed mean + supply gap
Horizon                                                                                                                   
6           1.168         1.798                                1.222                      1.329                      1.450
9           1.250         1.879                                1.511                      1.495                      1.650
12          1.353         1.949                                1.773                      1.654                      1.829
24          1.490         1.815                                2.147                      1.812                      1.939


## Direct positive-skew tests

These tests use positive 12-month average Kelly skewness as the evaluation sample. The pseudo-out-of-sample model estimates weights from all prior origins whose full forecast horizon has already been observed, then applies those real-time weights only to positive-skew months.


### 1. Contemporaneous gap composition


In [5]:
gap_use = positive_skew_sample & df["Headline_PCE"].notna() & df["Trimmed_Mean_PCE"].notna()
gap = df.loc[gap_use, "Headline_PCE"] - df.loc[gap_use, "Trimmed_Mean_PCE"]
component_cols = COMPONENT_COLS + (["Residual_Gap"] if "Residual_Gap" in df.columns else [])
component_names = COMPONENT_NAMES + (["Residual"] if "Residual_Gap" in df.columns else [])

rows = []
for label, col in zip(component_names, component_cols):
    contribution = df.loc[gap_use, col].mean()
    rows.append(
        {
            "Component": label,
            "Mean contribution": contribution,
            "Share of mean gap (%)": 100 * contribution / gap.mean(),
        }
    )

gap_decomposition = pd.DataFrame(rows)
print(f"Positive-skew observations: {int(gap_use.sum())}")
print(f"Mean headline - trimmed mean gap: {gap.mean():0.3f}")
print(format_table(gap_decomposition))


Positive-skew observations: 64
Mean headline - trimmed mean gap: 1.307
Component  Mean contribution  Share of mean gap (%)
   Demand              0.564                 43.116
   Supply              0.604                 46.230
Ambiguous              0.179                 13.726
 Residual             -0.040                 -3.071


### 2. Future headline-minus-trimmed error regression


In [6]:
regression_rows = []
for h in HORIZONS:
    target = f"Headline_next_{h}m_avg"
    use = positive_skew_sample & df[target].notna()
    y = df.loc[use, target] - df.loc[use, "Trimmed_Mean_PCE"]
    x = df.loc[use, COMPONENT_COLS]
    fit = ols_hac_fit(y, x, COMPONENT_NAMES, lag=h - 1)
    for term, coef, t_stat in zip(fit["terms"], fit["coef"], fit["t"]):
        if term == "Intercept":
            continue
        regression_rows.append(
            {
                "Horizon": h,
                "N": fit["n"],
                "Component": term,
                "Coef": coef,
                "HAC t-stat": t_stat,
                "R2": fit["r2"],
            }
        )

predictive_regressions = pd.DataFrame(regression_rows)
print(format_table(predictive_regressions))


 Horizon  N Component   Coef  HAC t-stat    R2
       6 59    Demand  1.423       3.320 0.740
       6 59    Supply -0.024      -0.051 0.740
       6 59 Ambiguous  2.964       9.706 0.740
       9 59    Demand  1.584       3.048 0.755
       9 59    Supply -0.430      -0.776 0.755
       9 59 Ambiguous  3.694       8.425 0.755
      12 59    Demand  1.669       2.661 0.753
      12 59    Supply -0.807      -1.345 0.753
      12 59 Ambiguous  4.234       7.564 0.753
      24 59    Demand  0.951       1.480 0.727
      24 59    Supply -1.258      -3.544 0.727
      24 59 Ambiguous  4.512       8.917 0.727


### 3. Expanding-window adjustment weights


In [7]:
def expanding_adjustment_forecasts(sample, train_sample, horizon, min_train=MIN_TRAIN):
    target = f"Headline_next_{horizon}m_avg"
    idx = np.arange(len(df))
    y_all = (df[target] - df["Trimmed_Mean_PCE"]).to_numpy(dtype=float)
    x_all = df[COMPONENT_COLS].to_numpy(dtype=float)
    tm_all = df["Trimmed_Mean_PCE"].to_numpy(dtype=float)
    sample_values = sample.to_numpy(dtype=bool)
    train_values = train_sample.to_numpy(dtype=bool)
    complete_train = np.isfinite(y_all) & np.all(np.isfinite(x_all), axis=1)

    rows = []
    for i in idx:
        if not sample_values[i]:
            continue
        if not np.isfinite(y_all[i]) or not np.isfinite(tm_all[i]) or not np.all(np.isfinite(x_all[i])):
            continue

        available_train = train_values & complete_train & ((idx + horizon) <= i)
        if int(available_train.sum()) < min_train:
            continue

        beta = fit_no_intercept(y_all[available_train], x_all[available_train])
        rows.append(
            {
                "idx": i,
                "Date": df.loc[i, "Date"],
                "Forecast": tm_all[i] + x_all[i] @ beta,
                "Beta Demand": beta[0],
                "Beta Supply": beta[1],
                "Beta Ambiguous": beta[2],
            }
        )
    return pd.DataFrame(rows)


benchmark_cols = {
    "Estimated D/S/A addback": None,
    "Trimmed mean": "Trimmed_Mean_PCE",
    "Core PCE": "Core_PCE",
    "Headline PCE": "Headline_PCE",
    "Trimmed mean + demand gap": "TM_plus_Demand",
}

weight_rows = []
performance_rows = []
for h in HORIZONS:
    oos = expanding_adjustment_forecasts(positive_skew_sample, all_months_sample, h)
    if oos.empty:
        continue

    eval_idx = oos["idx"].to_numpy(dtype=int)
    target = f"Headline_next_{h}m_avg"
    actual = df.loc[eval_idx, target].to_numpy(dtype=float)

    weight_rows.append(
        {
            "Horizon": h,
            "OOS N": len(oos),
            "First origin": oos["Date"].min().date(),
            "Last origin": oos["Date"].max().date(),
            "Mean beta demand": oos["Beta Demand"].mean(),
            "Mean beta supply": oos["Beta Supply"].mean(),
            "Mean beta ambiguous": oos["Beta Ambiguous"].mean(),
            "Last beta demand": oos["Beta Demand"].iloc[-1],
            "Last beta supply": oos["Beta Supply"].iloc[-1],
            "Last beta ambiguous": oos["Beta Ambiguous"].iloc[-1],
        }
    )

    for label, col in benchmark_cols.items():
        forecast = oos["Forecast"].to_numpy(dtype=float) if col is None else df.loc[eval_idx, col].to_numpy(dtype=float)
        valid = np.isfinite(actual) & np.isfinite(forecast)
        error = forecast[valid] - actual[valid]
        performance_rows.append(
            {
                "Horizon": h,
                "Measure": label,
                "N": int(error.size),
                "Bias": error.mean(),
                "MAE": np.mean(np.abs(error)),
                "RMSE": np.sqrt(np.mean(error**2)),
            }
        )

oos_weights = pd.DataFrame(weight_rows)
oos_performance = pd.DataFrame(performance_rows).sort_values(["Horizon", "RMSE"]).reset_index(drop=True)

print("Expanding-window weights estimated from all completed prior observations; evaluation is positive-skew origins only.")
print(format_table(oos_weights))
print()
print(format_table(oos_performance))


Expanding-window weights estimated from all completed prior observations; evaluation is positive-skew origins only.
 Horizon  OOS N First origin Last origin  Mean beta demand  Mean beta supply  Mean beta ambiguous  Last beta demand  Last beta supply  Last beta ambiguous
       6     56   2007-12-01  2023-01-01             1.023             0.493                1.003             1.015             0.653                1.472
       9     56   2007-12-01  2023-01-01             0.942             0.303                0.941             0.937             0.528                1.627
      12     56   2007-12-01  2023-01-01             0.829             0.121                0.829             0.805             0.401                1.711
      24     56   2007-12-01  2023-01-01             0.510            -0.289                0.299             0.136            -0.022                0.719

 Horizon                   Measure  N   Bias   MAE  RMSE
       6   Estimated D/S/A addback 56 -0.117 0.769 

## All months


In [8]:
all_results = evaluate(all_months_sample, measures)
print(format_table(all_results))


 Horizon                             Measure   N  Bias   MAE  RMSE
       6 Trimmed mean + demand + supply gaps 539 0.082 0.493 0.710
       6           Trimmed mean + demand gap 539 0.152 0.556 0.745
       6                            Core PCE 539 0.091 0.586 0.759
       6           Trimmed mean + supply gap 539 0.018 0.564 0.797
       6                        Trimmed mean 539 0.088 0.615 0.896
       9           Trimmed mean + demand gap 536 0.170 0.599 0.815
       9                            Core PCE 536 0.110 0.626 0.815
       9 Trimmed mean + demand + supply gaps 536 0.100 0.596 0.853
       9           Trimmed mean + supply gap 536 0.038 0.624 0.891
       9                        Trimmed mean 536 0.108 0.632 0.919
      12                            Core PCE 533 0.128 0.666 0.874
      12           Trimmed mean + demand gap 533 0.188 0.646 0.887
      12                        Trimmed mean 533 0.127 0.652 0.941
      12           Trimmed mean + supply gap 533 0.057 0.683 0

In [9]:
all_rmse_table = all_results.pivot(index="Horizon", columns="Measure", values="RMSE")
print(all_rmse_table.to_string(float_format=lambda x: f"{x:0.3f}"))


Measure  Core PCE  Trimmed mean  Trimmed mean + demand + supply gaps  Trimmed mean + demand gap  Trimmed mean + supply gap
Horizon                                                                                                                   
6           0.759         0.896                                0.710                      0.745                      0.797
9           0.815         0.919                                0.853                      0.815                      0.891
12          0.874         0.941                                0.982                      0.887                      0.973
24          1.055         0.993                                1.268                      1.070                      1.151
